# Cascade. How strict should the gate be?

Raising the Stage I threshold moves images from "passed" to "blocked (Grade 0)". Every image
above the operating point already has a Stage III prediction, so the upward sweep needs no new
inference. Lowering the threshold would admit images that were never segmented, so the sweep
cannot be extended downwards.

The QWK-optimal threshold is selected on test data and is shown only to illustrate the
trade-off; the cascade keeps the validation-derived operating point.

**Steps.**
1. Rebuild the cascade frame with the OOF-fitted thresholds.
2. Sweep the gate threshold from the operating point to 0.90.
3. Plot QWK and the two error counts against the threshold.

## 0. Configuration

In [ ]:
from pathlib import Path

# Inputs (all per-image CSVs):
#   data/test/test_grade.csv                image, grade (0..5) for all 58,689 test images
#   data/test/stage1_manifest.csv           stem, prob, pred_binary (Stage I routing; 1 = passed)
#   outputs/stage3/eval/oof_corn.csv        OOF predictions (stage3_grading/02)
#   outputs/stage3/eval/lgdrg_test_predictions.csv   ensemble predictions (stage3_grading/03)
DATA_ROOT = Path("data")
GT_CSV = DATA_ROOT / "test" / "test_grade.csv"
MANIFEST_CSV = DATA_ROOT / "test" / "stage1_manifest.csv"

OUT_ROOT = Path("outputs")
OOF_CSV = OUT_ROOT / "stage3" / "eval" / "oof_corn.csv"
PRED_CSV = OUT_ROOT / "stage3" / "eval" / "lgdrg_test_predictions.csv"
CASCADE_DIR = OUT_ROOT / "cascade"
CASCADE_DIR.mkdir(parents=True, exist_ok=True)

GATE_THRESHOLD = 0.2391715943813324     # frozen Stage I operating point

import json
import matplotlib.pyplot as plt
from hierarchiretina.stage3 import cascade as cas

## 1. Cascade frame

Uses the thresholds written by `01_end_to_end_evaluation` (identical to
`stage3_grading/02`).

In [ ]:
th = json.load(open(CASCADE_DIR / "corn_thresholds_consecutive.json"))
C = cas.build_cascade_frame(cas.load_ground_truth(GT_CSV), cas.load_stage1_manifest(MANIFEST_CSV),
                            cas.load_lgdrg_predictions(PRED_CSV), th["t"], th["tg"])

## 2. Sweep

In [ ]:
sweep = cas.gate_sweep(C, GATE_THRESHOLD)
sweep.to_csv(CASCADE_DIR / "gate_sweep.csv", index=False)
best = sweep.loc[sweep.qwk.idxmax()]
print(f"operating point {GATE_THRESHOLD:.4f}: QWK {sweep.qwk.iloc[0]:.4f}")
print(f"QWK maximum (test-selected, not used): threshold {best.threshold:.3f}, QWK {best.qwk:.4f}, "
      f"DR blocked {int(best.dr_lost):,}")
sweep.round(4)

## 3. Figure

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(9, 3.4))
ax[0].plot(sweep.threshold, sweep.qwk, marker="o", ms=3)
ax[0].axvline(GATE_THRESHOLD, ls="--", c="crimson", label="operating point")
ax[0].set_xlabel("Stage I threshold"); ax[0].set_ylabel("end-to-end QWK"); ax[0].legend()
ax[1].plot(sweep.threshold, sweep.dr_lost, marker="o", ms=3, label="DR cases blocked")
ax[1].plot(sweep.threshold, sweep.healthy_in, marker="s", ms=3, label="healthy eyes admitted")
ax[1].axvline(GATE_THRESHOLD, ls="--", c="crimson")
ax[1].set_xlabel("Stage I threshold"); ax[1].set_ylabel("images"); ax[1].legend()
fig.tight_layout()